Investigate why certain color distributions look off. 

In [ ]:
import argparse
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from warptemplate import WarpfitTemplateLoader, add_warpclasses, register_all
register_all()


In [ ]:
class_name = 'SN Ia-91T'

In [ ]:
def get_color(model, z, band1, band2):

    rest_phase = model.source.peakphase(band1)
    inz = float(model.get('z'))

    model.set(z=z)
    obs_phase = rest_phase * (1 + z)
    color = model.bandmag(band1, "ab", obs_phase) - model.bandmag(band2, "ab", obs_phase)
    model.set(z=inz)
    return color

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sncosmo

def compare_sncosmo_bands(models, bands=('ztfg', 'ztfr', 'ztfi'),
                          time_range=(-20, 50), n_time=200,
                          z=0.1, labels=None, figsize=(12, 4),
                          absolute=False, **plot_kwargs):
    """
    Plot and compare sncosmo model light curves in specified bands.
    
    Parameters
    ----------
    models : list of sncosmo.Model or str
        Model instances or model names
    bands : tuple of str
        Bandpass names (must be registered in sncosmo)
    time_range : tuple
        (min, max) time relative to t0 in days
    n_time : int
        Number of time points
    z : float
        Redshift for cosmological distance and K-corrections
    labels : list of str, optional
        Legend labels; defaults to model source names
    absolute : bool
        Plot absolute magnitudes (requires known distance from z)
    **plot_kwargs : passed to ax.plot()
    
    Returns
    -------
    fig, axes : matplotlib Figure and Axes array
    """
    
    model_list = [sncosmo.Model(m) if isinstance(m, str) else m for m in models]
    n_models = len(model_list)
    
    if labels is None:
        labels = [getattr(m.source, 'name', f"model_{i}") 
                  for i, m in enumerate(model_list)]
    
    # Common time grid in observer frame
    time_obs = np.linspace(time_range[0], time_range[1], n_time)
    
    # Setup: one panel per band
    fig, axes = plt.subplots(1, len(bands), figsize=figsize, sharey=True)
    if len(bands) == 1:
        axes = [axes]
    
    colors = plt.cm.tab10(np.linspace(0, 0.9, n_models))
    
    for ax, band in zip(axes, bands):
        # Verify band exists
        try:
            sncosmo.get_bandpass(band)
        except Exception:
            raise ValueError(f"Band '{band}' not registered in sncosmo")
        
        for j, (model, label) in enumerate(zip(model_list, labels)):
            # Clone to avoid modifying caller's model
            m = model
            
            # Set standard parameters if not already set
            m.set(z=z, t0=0.)

            try:
                # Compute AB magnitudes
                mag = m.bandmag(band, 'ab', time_obs)
                
                # Convert to absolute if requested
                if absolute:
                    # Simple H0=70 flat LCDM; for precision work use astropy.cosmology
                    from astropy.cosmology import FlatLambdaCDM
                    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
                    dm = cosmo.distmod(z).value
                    mag = mag - dm
                
                # Mask invalid (model undefined) regions
                valid = np.isfinite(mag)
                ax.plot(time_obs[valid], mag[valid], color=colors[j], 
                       label=label if band == bands[0] else "", **plot_kwargs)
            except ValueError:
                continue
        
        ax.set_xlabel('Phase [days]')
        ax.set_title(band)
        ax.invert_yaxis()  # Brighter = higher
        ax.grid(True, alpha=0.3)
    
    axes[0].set_ylabel('Apparent AB mag' if not absolute else 'Absolute AB mag')
    axes[0].legend(loc='lower right', frameon=False, fontsize=8)
    
    plt.tight_layout()
    return fig, axes




In [ ]:
# Generate templates in three modes
warploader = WarpfitTemplateLoader(
        "/Users/jnordin/data/models/sncosmo/warpmod/v5",
        version=5,
        suffix='',
    )

In [ ]:
rawtemp = warploader.get_templates(
                fitclass=class_name,
                exclude_input=[],
                template_selection='all',
                snbasis_selection='all',
                color_mode=None,
            )

In [ ]:
hartemp = warploader.get_templates(
                fitclass=class_name,
                exclude_input=[],
                template_selection='all',
                snbasis_selection='all',
                color_mode='harmonize',
            )

In [ ]:
drawtemp = warploader.get_templates(
                fitclass=class_name,
                exclude_input=[],
                template_selection='all',
                snbasis_selection='all',
                color_mode='draw',
            )

In [ ]:
snid = 20

In [ ]:
# First step, find a suitable object to look at 
rawtemp[snid]

In [ ]:
hartemp[snid]

In [ ]:
drawtemp[snid]

In [ ]:
get_color( rawtemp[snid]['model'], 0, 'ztfg', 'ztfr')

In [ ]:
get_color( hartemp[snid]['model'], 0, 'ztfg', 'ztfr')

In [ ]:
get_color( drawtemp[snid]['model'], 0, 'ztfg', 'ztfr')

In [ ]:
# Example usage:
# fig, axes = compare_sncosmo_bands(
#     ['salt2', 'snana-2004fe'],
#     bands=('ztfg', 'ztfr', 'ztfi'),
#     z=0.05,
#     time_range=(-15, 40)
# )

In [ ]:
snid = 8
fig, axes = compare_sncosmo_bands(
     [rawtemp[snid]['model'], hartemp[snid]['model'], drawtemp[snid]['model']],
     bands=('ztfg', 'ztfr', 'ztfi'),
     z=0.0,
     time_range=(-15, 40), 
    labels = ['raw', 'harmonized', 'draw'],
 )

In [ ]:
fig, axes = compare_sncosmo_bands(
     [rawtemp[snid]['model'], hartemp[snid]['model'], drawtemp[snid]['model']],
     bands=('ztfg', 'ztfr', 'ztfi'),
     z=0.1,
     time_range=(-15, 40), 
    labels = ['raw', 'harmonized', 'draw'],
 )

In [ ]:
# Go through the templates and compare colors
lr, ld = [], []
ztest = 0.08
for k in range(len(rawtemp)):
    try:
        cr = get_color( rawtemp[k]['model'], ztest, 'ztfg', 'ztfr')
        ch = get_color( hartemp[k]['model'], ztest, 'ztfg', 'ztfr')
        cd = get_color( drawtemp[k]['model'], ztest, 'ztfg', 'ztfr')
        cr0 = get_color( rawtemp[k]['model'], 0, 'ztfg', 'ztfr')
        ch0 = get_color( hartemp[k]['model'], 0, 'ztfg', 'ztfr')
        cd0 = get_color( drawtemp[k]['model'], 0, 'ztfg', 'ztfr')
        print(k, cr, ch, cd, cr0, ch0, cd0, cr-cd)
        lr.append(cr)
        ld.append(cd)
    except ValueError:
        continue

In [ ]:
plt.hist(lr)

In [ ]:
plt.hist(ld)

In [ ]:
# Go through the templates and check for redshifted, drawn colors which are too blue.
# Maybe there are particular sne or templates in there?
ztest = 0.08
sne, tem = [], []
for k in range(len(rawtemp)):
    try:
        cd = get_color( drawtemp[k]['model'], ztest, 'ztfg', 'ztfr')
        if cd<-0.3:
            print( drawtemp[k]['basis_sn'], drawtemp[k]['template_sn'] )
            sne.append( drawtemp[k]['basis_sn'] )
            tem.append( drawtemp[k]['template_sn'] )
    except ValueError:
        continue

In [ ]:
sne

In [ ]:
allsn = [
    drawtemp[k]['basis_sn']
    for k in range(len(rawtemp))
]

In [ ]:
temsn = [
    drawtemp[k]['template_sn']
    for k in range(len(rawtemp))
]

In [ ]:
temsn

In [ ]:
from collections import Counter

In [ ]:
Counter(sne)

In [ ]:
Counter(allsn)

In [ ]:
Counter(tem)

In [ ]:
Counter(temsn)